# Классификация ботов по логам событий — единый пайплайн

Метрика: precision при recall >= 0.70. Валидация — expanding window по времени,
тест лежит строго позже трейна, поэтому случайное разбиение завышает скор.

## Устройство пайплайна

1. **`FeatureEngineer`** — единственный источник признаков. Пять групп
   (`basic`, `temporal`, `bigram`, `graph`, `ua_semantic`), включаются
   конфигурацией `FEATURE_GROUPS`. Каждая группа проверяется отдельной
   гипотезой и включается в модель только если даёт прирост.
2. **Равные условия** — на каждой конфигурации признаков CatBoost, LightGBM и
   XGBoost проходят полный цикл Optuna (30 проб, TPE, seed 0). Параметры
   одной модели не переиспользуются для другой конфигурации.
3. **Лестница гипотез** — конфигурации проверяются последовательно, каждая
   откатывается, если прирост pooled OOF не превышает `MIN_GAIN`.
4. **Ансамблирование** — rank-average блендинг против вложенного стэкинга,
   выбор по pooled OOF.

## Окно наблюдения — жёсткое требование

Признаки считаются **только** по событиям, где
`window_start_ts <= event_ts < window_end_ts`. Все 7 928 событий `captcha_shown`
лежат строго за окном: без фильтра они дают почти идеальное разделение на
train (доля ботов 77 %), но в test captcha-событий нет вообще, и модель на
таком признаке молча деградирует. Поэтому проверяем ассертом.

In [1]:
import re
import time
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

import optuna
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from metric import precision_at_recall

warnings.filterwarnings("ignore")

SEED = 0
N_TRIALS = 30
BIGRAM_N = 15
SESSION_GAP_SEC = 1800
MIN_GAIN = 0.005
PROMOTE_THRESHOLD = 0.75

VALID_FOLDS = [
    ("2026-04-13", "2026-04-15"),
    ("2026-04-15", "2026-04-17"),
    ("2026-04-17", "2026-04-20"),
]

FEATURE_GROUPS = {
    "basic": True,
    "temporal": False,
    "bigram": False,
    "graph": False,
    "ua_semantic": False,
}

GRAPH_NODES = {
    "item": ["item_id"],
    "catloc": ["item_category", "item_location"],
}

/home/alexander/Work/bot_detection_challenge/venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv(
    "data/train.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
test = pd.read_csv(
    "data/test.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

EVENT_NAMES = sorted(events.event_name.unique())
PLATFORMS = ["android", "iphone", "web"]

y = train.target.values

BOT_UA = re.compile(
    r"headless|curl/|scrapy|python-requests|python-urllib|go-http-client|wget|phantomjs|selenium",
    re.IGNORECASE,
)

# `platform` записан в 11 вариантах регистра одного и того же значения.
events["platform"] = events.platform.str.lower().str.strip()
events["ua_is_bot"] = events.user_agent.fillna("").str.contains(BOT_UA)

print(f"train: {train.shape}, test: {test.shape}, events: {events.shape}")
print(f"ботов в train: {int(train.target.sum())} из {len(train)} ({train.target.mean():.2%})")
print(f"общих cookie между train и test: {len(set(train.cookie_id) & set(test.cookie_id))}")
print(f"типов событий: {len(EVENT_NAMES)}, платформ после lower(): {sorted(events.platform.unique())}")

train: (11091, 5), test: (4909, 4), events: (328905, 15)
ботов в train: 899 из 11091 (8.11%)
общих cookie между train и test: 0
типов событий: 10, платформ после lower(): ['android', 'desktop', 'ios', 'iphone', 'web']


In [3]:
def events_in_window(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """События, попавшие в окно наблюдения своей куки.

    Полные дубликаты строк — это повторная доставка одного события, а не два
    разных: наивный groupby посчитал бы их дважды, исказив и объём, и тайминги.
    """
    merged = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
    ).drop_duplicates()
    inside = (merged.event_ts >= merged.window_start_ts) & (merged.event_ts < merged.window_end_ts)
    return merged[inside]


ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)

assert int((ev_train.event_name == "captcha_shown").sum()) == 0, "captcha внутри окна train"
assert int((ev_test.event_name == "captcha_shown").sum()) == 0, "captcha внутри окна test"

print(f"событий train: {len(events):,} -> в окне {len(ev_train):,}")
print(f"событий test:  {len(events):,} -> в окне {len(ev_test):,}")
print("captcha_shown внутри окна: 0 (train) и 0 (test) — как и должно быть")

событий train: 328,905 -> в окне 195,482
событий test:  328,905 -> в окне 88,348
captcha_shown внутри окна: 0 (train) и 0 (test) — как и должно быть


## `FeatureEngineer` — единственный источник признаков

Один класс вместо прежних `build_features` и `build_features_v2`. Группы
включаются словарём `FEATURE_GROUPS` и добавляются в фиксированном порядке:
basic → temporal → bigram → graph → ua_semantic.

Разделение `fit` / `transform` существенно по трём причинам:

* **словарь биграмм** должен строиться только по train-части текущего фолда,
  иначе валидация подглядывает в определение признака;
* **target encoding** UA-групп считается на train-части фолда;
* **граф** считается по всем событиям файла: это трандуктивно, но безопасно —
  метки не используются, а пул соседей одинаков для train и test.

Графовые признаки нормализованы на медианную степень узла в пуле. Без
нормализации они дают высокий AUC исключительно из-за размера пула: валидация
всегда считается против 11 091 трейновой куки, а лидерборд — против 4 909
тестовых, и «сырые» счётчики соседей расходятся в ~2 раза. Нормированные
признаки инвариантны к размеру пула, поэтому переносятся на тест.

In [4]:
class FeatureEngineer:
    """Единый генератор признаков с настраиваемыми группами."""

    BASIC_DROP = {
        # Доли пропусков — это копия event_name, а не поведение.
        "frac_seller_page_view",
        "frac_search_results_view",
        "frac_search_events",
        "search_query_repeat_rate",
        "nuniq_search_query",
        "nuniq_seller_type",
        "frac_contact_message_sent",
        "frac_contact_phone_show",
        # Техника, а не поведение: по permutation importance только мешали.
        "events_per_sec",
        "span_sec",
        "n_span_fill",
        "frac_web",
        "nuniq_ua_bot",
        "is_static_cursor",
        "mean_search_query_length",
        "ua_is_headless",
        "ua_is_bot_flag",
        "ua_mean_length",
        "ua_mean_n_tokens",
        "ua_nuniq_browser",
        "ua_nuniq_os",
        "ua_browser_top",
        "ua_os_top",
        "ua_device_top",
        "frac_android",
        "frac_iphone",
        # Тождественно нулевой: все captcha_shown лежат строго за окном.
        # Держим вне модели ещё и потому, что без фильтра по окну этот столбец
        # снова становится почти идеальным разделителем (доля ботов 77 %).
        "frac_captcha_shown",
    }

    UA_AUTOMATION = re.compile(
        r"headless|curl/|scrapy|python-requests|python-urllib|go-http-client|"
        r"wget|phantomjs|selenium|java/|okhttp|libwww|http-client",
        re.IGNORECASE,
    )

    def __init__(self, groups: dict | None = None,
                 bigram_n: int = BIGRAM_N,
                 session_gap_sec: int = SESSION_GAP_SEC,
                 graph_nodes: dict | None = None):
        self.groups = dict(FEATURE_GROUPS if groups is None else groups)
        self.bigram_n = bigram_n
        self.session_gap_sec = session_gap_sec
        self.graph_nodes = dict(GRAPH_NODES if graph_nodes is None else graph_nodes)
        self.bigram_vocab: list[str] = []
        self.graph_degree: dict[str, pd.Series] = {}
        self.graph_ref: dict[str, float] = {}
        self.ua_maps: dict[str, tuple] = {}

    @property
    def active(self) -> list[str]:
        return [name for name in FEATURE_GROUPS if self.groups.get(name)]

    def fit(self, events_all: pd.DataFrame, meta: pd.DataFrame,
            y: np.ndarray | None = None) -> "FeatureEngineer":
        """Всё, что зависит от обучающей выборки: словари и референсы.

        `events_all` — полная таблица событий без фильтра по окну. Граф обязан
        строиться по ней, иначе пул соседей у train и test разойдётся.
        """
        if self.groups.get("bigram"):
            fit_ids = meta.cookie_id if y is None else meta.cookie_id[: len(y)]
            window = events_in_window(events_all, meta)
            self.bigram_vocab = self._bigram_vocabulary(
                self._event_pairs(window[window.cookie_id.isin(fit_ids)])
            )
        if self.groups.get("graph"):
            self._fit_graph(events_all)
        if self.groups.get("ua_semantic") and y is not None:
            self._fit_ua_semantic(events_in_window(events_all, meta), meta, y)
        return self

    def transform(self, events_window: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
        """Признаки для перечисленных в `meta.cookie_id` кук."""
        index = pd.Index(meta.cookie_id, name="cookie_id")
        parts: list[pd.DataFrame] = []

        if self.groups.get("basic"):
            parts.append(self._basic(events_window, index))
        if self.groups.get("temporal"):
            parts.append(self._temporal(events_window, index))
        if self.groups.get("bigram"):
            parts.append(self._bigrams(events_window, index))
        if self.groups.get("graph"):
            parts.append(self._graph(events_window, index))
        if self.groups.get("ua_semantic"):
            parts.append(self._ua_semantic(events_window, index))

        X = pd.concat(parts, axis=1)
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return X

    # ------------------------------------------------------------------ basic

    @staticmethod
    def _entropy(values: pd.Series) -> float:
        shares = values.value_counts(normalize=True)
        return float(-(shares * np.log(shares)).sum())

    @staticmethod
    def _gap_entropy(gaps: pd.Series) -> float:
        counts = pd.cut(
            gaps.dropna(), bins=np.linspace(0, 600, 21), duplicates="drop"
        ).value_counts(normalize=True)
        counts = counts[counts > 0]
        return float(-(counts * np.log(counts)).sum())

    def _basic(self, ev: pd.DataFrame, index: pd.Index) -> pd.DataFrame:
        ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort").copy()
        ev["gap"] = ev.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
        ev["hod"] = ev.event_ts.dt.hour + ev.event_ts.dt.minute / 60

        by_cookie = ev.groupby("cookie_id")
        out = pd.DataFrame(index=index)

        out["n_events"] = by_cookie.size()
        out["events_per_sec"] = out.n_events / (by_cookie.event_ts.agg(
            lambda s: (s.max() - s.min()).total_seconds()).fillna(0.0) + 60.0)
        out["span_sec"] = by_cookie.event_ts.agg(lambda s: (s.max() - s.min()).total_seconds())
        out["gap_median"] = by_cookie.gap.median()
        out["gap_std"] = by_cookie.gap.std()
        out["gap_min"] = by_cookie.gap.min()
        out["gap_p90"] = by_cookie.gap.quantile(0.90)
        out["frac_gap_zero"] = by_cookie.gap.apply(lambda s: float((s == 0).mean()))
        out["frac_gap_lt1s"] = by_cookie.gap.apply(lambda s: float(s.between(0.01, 1).mean()))
        out["frac_gap_gt60s"] = by_cookie.gap.apply(lambda s: float((s > 60).mean()))
        out["gap_cv"] = out.gap_std / (out.gap_median + 1e-6)
        out["gap_entropy"] = by_cookie.gap.apply(self._gap_entropy)

        for column in ("item_id", "item_category", "item_location", "seller_type",
                       "search_query", "search_page", "platform", "user_agent"):
            out[f"nuniq_{column}"] = by_cookie[column].nunique()

        for platform in PLATFORMS:
            share = ev.assign(t=ev.platform == platform).groupby("cookie_id")["t"].mean()
            out[f"frac_{platform}"] = share
        for name in EVENT_NAMES:
            share = ev.assign(t=ev.event_name == name).groupby("cookie_id")["t"].mean()
            out[f"frac_{name}"] = share
        out["event_entropy"] = by_cookie.event_name.apply(self._entropy)

        night = ev.assign(t=ev.hod.between(1, 5)).groupby("cookie_id")["t"].mean()
        out["frac_night"] = night
        out["mean_hod"] = by_cookie.hod.mean()
        out["std_hod"] = by_cookie.hod.std()

        out["pointer_std"] = by_cookie.pointer_x.std()
        out["pointer_nuniq"] = by_cookie.pointer_x.nunique()
        out["ua_is_bot"] = by_cookie.ua_is_bot.mean()
        out["nuniq_ua_bot"] = by_cookie.ua_is_bot.sum()

        out = out.drop(columns=[c for c in out.columns if c in self.BASIC_DROP])
        out.columns = [re.sub(r"[^0-9A-Za-z_]+", "_", str(c)) for c in out.columns]
        return out

    # -------------------------------------------------------------- temporal

    def _sessions(self, ev: pd.DataFrame) -> pd.DataFrame:
        ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort").copy()
        gap = ev.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
        ev["gap"] = gap
        starts = (gap.isna() | (gap > self.session_gap_sec)).astype(int)
        ev["session"] = starts.groupby(ev.cookie_id).cumsum()
        return ev

    def _temporal(self, ev: pd.DataFrame, index: pd.Index) -> pd.DataFrame:
        ev = self._sessions(ev)
        # Разрыв внутри сессии, а не внутри куки: сессия делится на «пакеты» активности.
        ev["within_gap"] = ev.groupby(["cookie_id", "session"])["event_ts"].diff().dt.total_seconds()
        within = ev.groupby(["cookie_id", "session"])["within_gap"].agg(["std", "mean"])
        by_cookie = ev.groupby("cookie_id")

        out = pd.DataFrame(index=index)
        out["n_sessions"] = by_cookie.session.nunique()
        out["max_session_len"] = ev.groupby(["cookie_id", "session"]).size() \
            .groupby(level="cookie_id").max()
        # Среднее число событий в сессии — не то же самое, что 1/n_sessions:
        # именно эта величина даёт AUC 0.711, лучший одиночный признак задачи.
        out["mean_events_per_session"] = by_cookie.size() / out.n_sessions.replace(0, np.nan)

        out["anchor_first_to_item"] = self._anchor(ev, None, "item_view")
        out["anchor_item_to_contact"] = self._anchor(ev, "item_view", "contact_.*")
        out["has_item_view"] = ev.assign(t=ev.event_name == "item_view") \
            .groupby("cookie_id")["t"].any().astype(float)
        out["has_contact"] = ev.assign(t=ev.event_name.str.startswith("contact_")) \
            .groupby("cookie_id")["t"].any().astype(float)

        # Вариативность скорости внутри сессии. Знак не разворачиваем: деревья
        # инвариантны к монотонным преобразованиям, поэтому AUC 0.37 не хуже 0.63.
        out["cv_intra_session"] = (within["std"] / within["mean"].clip(lower=1e-9)) \
            .groupby(level="cookie_id").mean()
        out["gap_std_within"] = ev.groupby("cookie_id").within_gap.std()
        out["max_zero_gap_run"] = ev.assign(z=(ev.gap == 0).astype(int)) \
            .groupby(["cookie_id", "session"]).z.sum().groupby(level="cookie_id").max()
        # Медиана и стандартное отклонение разрывов уже есть в basic как
        # `gap_median` и `gap_std`, а доля нулевых разрывов — как `frac_gap_zero`.
        # Дубли не добавляем: они не дают новой информации и только шумят отбор.
        return out

    @staticmethod
    def _anchor(ev: pd.DataFrame, start_event: str | None,
                target_pattern: str) -> pd.Series:
        """Время от первого `start_event` до первого события, подходящего под шаблон.

        `start_event=None` означает первое событие куки как есть.
        """
        if start_event is None:
            start = ev.groupby("cookie_id").event_ts.min()
        else:
            start = ev[ev.event_name == start_event].groupby("cookie_id").event_ts.min()
        target = ev[ev.event_name.str.match(target_pattern)] \
            .groupby("cookie_id").event_ts.min()
        return (target - start).dt.total_seconds()

    # ---------------------------------------------------------------- bigrams

    @staticmethod
    def _event_pairs(ev: pd.DataFrame) -> pd.DataFrame:
        """Пары «предыдущее → текущее» событие внутри каждой куки."""
        ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort")
        previous = ev.event_name.shift()
        has_prev = previous.notna()
        return pd.DataFrame({
            "cookie_id": ev.cookie_id.to_numpy()[has_prev.to_numpy()],
            "bigram": (previous[has_prev].astype(str)
                       + " -> " + ev.event_name[has_prev].astype(str)).to_numpy(),
        })

    def _bigram_vocabulary(self, pairs: pd.DataFrame) -> list[str]:
        counts = pairs.bigram.value_counts()
        return list(counts.index[: self.bigram_n])

    def _bigrams(self, ev: pd.DataFrame, index: pd.Index) -> pd.DataFrame:
        pairs = self._event_pairs(ev)
        pairs = pairs[pairs.bigram.isin(self.bigram_vocab)]
        counts = pd.crosstab(pairs.cookie_id, pairs.bigram)
        totals = pairs.groupby("cookie_id").size()
        out = pd.DataFrame(index=index)
        for i, bigram in enumerate(self.bigram_vocab):
            share = counts[bigram] / totals if bigram in counts else 0.0
            out[f"bg_{i:02d}"] = share.reindex(index).fillna(0.0)
        return out

    # ------------------------------------------------------------------ graph

    @staticmethod
    def _node_key(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
        key = frame[columns[0]].astype(str)
        for column in columns[1:]:
            key = key + "|" + frame[column].astype(str)
        return key

    def _fit_graph(self, events_all: pd.DataFrame) -> None:
        """Степень узла = сколько разных кук с этим товаром встречалось в файле.

        Считается по всем событиям без фильтра по окну: пул должен быть один и тот
        же для train и test, иначе нормализация не спасает от сдвига.
        """
        for spec, columns in self.graph_nodes.items():
            pairs = events_all.dropna(subset=columns).drop_duplicates(
                subset=["cookie_id"] + columns
            )
            keys = self._node_key(pairs, columns)
            degree = keys.groupby(keys.values).size()
            self.graph_degree[spec] = degree
            self.graph_ref[spec] = float(degree.median())

    def _graph(self, ev: pd.DataFrame, index: pd.Index) -> pd.DataFrame:
        out = pd.DataFrame(index=index)
        for spec, columns in self.graph_nodes.items():
            pairs = ev.dropna(subset=columns).drop_duplicates(
                subset=["cookie_id"] + columns
            )
            if pairs.empty:
                for suffix in ("mean_rel", "max_rel"):
                    out[f"g_{spec}_{suffix}"] = 0.0
                continue
            degree = self._node_key(pairs, columns).map(self.graph_degree[spec])
            relative = degree / self.graph_ref[spec]
            grouped = pd.Series(relative.to_numpy(), index=pairs.cookie_id.to_numpy()) \
                .groupby(level=0).agg(["mean", "max"])
            out[f"g_{spec}_mean_rel"] = grouped["mean"].reindex(index).fillna(0.0)
            out[f"g_{spec}_max_rel"] = grouped["max"].reindex(index).fillna(0.0)
        return out

    # ----------------------------------------------------------- ua_semantic

    @staticmethod
    def _ua_family(user_agent: str) -> tuple[str, str]:
        if not isinstance(user_agent, str) or not user_agent:
            return ("none", "none")
        if FeatureEngineer.UA_AUTOMATION.search(user_agent):
            return ("automation", "automation")
        browser = "other"
        for pattern, name in ((r"HeadlessChrome", "headlesschrome"), (r"Edg", "edge"),
                              (r"OPR", "opera"), (r"Chrome", "chrome"), (r"Firefox", "firefox"),
                              (r"Safari", "safari")):
            if re.search(pattern, user_agent, re.IGNORECASE):
                browser = name
                break
        operating = "other"
        for pattern, name in ((r"Windows", "windows"), (r"Mac OS|Macintosh", "macos"),
                              (r"Android", "android"), (r"iPhone|iPad|iOS", "ios"),
                              (r"X11|Linux", "linux")):
            if re.search(pattern, user_agent, re.IGNORECASE):
                operating = name
                break
        return (browser, operating)

    def _fit_ua_semantic(self, ev: pd.DataFrame, meta: pd.DataFrame,
                         y: np.ndarray) -> None:
        """Сглаженное target encoding по browser и os, посчитанное на train-части.

        Метки берутся из `y`, выровненного по `meta`, поэтому функция не зависит
        от глобальных таблиц. Сглаживание к 20 наблюдениям удерживает редкие
        группы от переобучения: `automation` встречается часто, но `safari` — 192
        куки, и без сглаживания такой код запоминал бы шум.
        """
        target = pd.Series(np.asarray(y, dtype=float), index=pd.Index(meta.cookie_id))
        families = ev.user_agent.map(self._ua_family)
        frame = pd.DataFrame({
            "cookie_id": ev.cookie_id.to_numpy(),
            "browser": [f[0] for f in families],
            "os": [f[1] for f in families],
        }).drop_duplicates(subset=["cookie_id"])
        frame["target"] = frame.cookie_id.map(target)
        frame = frame[frame.target.notna()]
        prior = float(np.asarray(y).mean())
        for column in ("browser", "os"):
            stats = frame.groupby(column).target.agg(["sum", "size"])
            smoothed = (stats["sum"] + 20.0 * prior) / (stats["size"] + 20.0)
            self.ua_maps[column] = smoothed.to_dict()

    def _ua_semantic(self, ev: pd.DataFrame, index: pd.Index) -> pd.DataFrame:
        families = ev.user_agent.map(self._ua_family)
        frame = pd.DataFrame({
            "cookie_id": ev.cookie_id.to_numpy(),
            "browser": families.map(lambda f: f[0]).to_numpy(),
            "os": families.map(lambda f: f[1]).to_numpy(),
        })
        out = pd.DataFrame(index=index)
        for column in ("browser", "os"):
            encoded = frame[column].map(self.ua_maps.get(column, {}))
            grouped = pd.Series(encoded.to_numpy(), index=frame.cookie_id.to_numpy())
            out[f"ua_te_{column}"] = grouped.groupby(level=0).mean().reindex(index).fillna(0.0)
        out["ua_frac_automation"] = frame.assign(
            t=frame.browser == "automation").groupby("cookie_id")["t"].mean().reindex(index).fillna(0.0)
        return out

## Честная валидация: expanding window CV

Окна в train: 2026-04-06 … 2026-04-20 (14 суточных). Окна в test:
2026-04-20 … 2026-04-27. `cookie_id` между ними не пересекаются.
Фолд `k` обучается на всём, что строго раньше `window_start_ts` фолда.

In [5]:
root = train.window_start_ts


def fold_masks(start: str, end: str) -> tuple[np.ndarray, np.ndarray]:
    """Маски (train, valid) для одного временного окна."""
    window_start, window_end = pd.Timestamp(start), pd.Timestamp(end)
    train_mask = (root < window_start).values
    valid_mask = ((root >= window_start) & (root < window_end)).values
    return train_mask, valid_mask


def to_rank(scores: np.ndarray) -> np.ndarray:
    """Нормированные порядковые ранги: ничьи метрике делить нельзя."""
    return rankdata(scores, method="ordinal") / len(scores)


def fit_model(factory, X: pd.DataFrame, y: np.ndarray,
              train_mask: np.ndarray, eval_mask: np.ndarray | None):
    """Обучает модель; `eval_mask` нужен только для early stopping."""
    model = factory()
    if eval_mask is None:
        if getattr(model, "early_stopping_rounds", None):
            model.set_params(early_stopping_rounds=None)
        model.fit(X[train_mask], y[train_mask])
        return model
    model.fit(X[train_mask], y[train_mask], eval_set=[(X[eval_mask], y[eval_mask])])
    return model


def cv_evaluate(factory, X: pd.DataFrame, y: np.ndarray, name: str = "model") -> dict:
    """Per-fold таблица, gap и pooled OOF для одного завода."""
    per_fold, oof_raw, oof_rank, oof_y = [], [], [], []

    for i, (start, end) in enumerate(VALID_FOLDS, 1):
        train_mask, valid_mask = fold_masks(start, end)
        model = fit_model(factory, X, y, train_mask, valid_mask)
        p_valid = model.predict_proba(X[valid_mask])[:, 1]
        p_train = model.predict_proba(X[train_mask])[:, 1]
        per_fold.append({
            "fold": i,
            "n_valid": int(valid_mask.sum()),
            "n_bots": int(y[valid_mask].sum()),
            "P@R0.7": precision_at_recall(y[valid_mask], p_valid),
            "AUC": roc_auc_score(y[valid_mask], p_valid),
            "train P@R0.7": precision_at_recall(y[train_mask], p_train),
        })
        oof_raw.append(p_valid)
        oof_rank.append(to_rank(p_valid))
        oof_y.append(y[valid_mask])

    table = pd.DataFrame(per_fold).set_index("fold")
    table["gap"] = table["train P@R0.7"] - table["P@R0.7"]
    oof_raw = np.concatenate(oof_raw)
    oof_rank = np.concatenate(oof_rank)
    oof_y = np.concatenate(oof_y)
    return {
        "name": name,
        "table": table,
        "pooled_oof": precision_at_recall(oof_y, oof_rank),
        "oof_raw": oof_raw,
        "oof_rank": oof_rank,
        "oof_y": oof_y,
    }


def bootstrap_pooled(result: dict, n: int = 500, seed: int = SEED):
    """90 %-интервал pooled OOF — оценка собственного шума метрики."""
    rng = np.random.default_rng(seed)
    p, y_true = result["oof_rank"], result["oof_y"]
    draws = rng.integers(0, len(y_true), size=(n, len(y_true)))
    values = np.array([precision_at_recall(y_true[idx], p[idx]) for idx in draws])
    return np.percentile(values, [5, 50, 95])

## Модели и поисковое пространство

Все три бустинга получают одинаковый бюджет: 30 проб Optuna, TPE, seed 0,
objective — средний P@R0.7 по трём внешним фолдам.

Пространства расширены против прежних: CatBoost и LightGBM были зажаты
(`depth` 3–6 и 3–7, `learning_rate` до 0.15, `min_data_in_leaf` от 50). Зажатая
регуляризация объясняет, почему XGBoost при равном бюджете догонял остальных.
Теперь глубина доходит до 12, `learning_rate` до 0.2, а нижняя граница
размера листа опущена до 10 — регуляризация переносится на L2 и отбор признаков.

In [6]:
class QuietXGBClassifier(XGBClassifier):
    """XGBoost, который не печатает оценку на каждом дереве.

    Дефолт `verbose=True` в sklearn-обвязке не связан с `verbosity=0` из
    конструктора, поэтому без подкласса в вывод ноутбука попадают тысячи строк.
    """

    def fit(self, X, y, **kwargs):
        kwargs.setdefault("verbose", False)
        return super().fit(X, y, **kwargs)


def make_catboost(params: dict, iterations: int | None = None):
    def factory():
        return CatBoostClassifier(
            iterations=iterations or params["iterations"],
            learning_rate=params["learning_rate"],
            depth=params["depth"],
            l2_leaf_reg=params["l2_leaf_reg"],
            min_data_in_leaf=params["min_data_in_leaf"],
            rsm=params["rsm"],
            verbose=0,
            random_state=SEED,
            early_stopping_rounds=50,
        )
    return factory


def make_lightgbm(params: dict, n_estimators: int | None = None):
    def factory():
        return LGBMClassifier(
            n_estimators=n_estimators or params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            num_leaves=params["num_leaves"],
            lambda_l1=params["lambda_l1"],
            lambda_l2=params["lambda_l2"],
            min_child_samples=params["min_child_samples"],
            feature_fraction=params["feature_fraction"],
            min_gain_to_split=params["min_gain_to_split"],
            verbose=-1,
            random_state=SEED,
            early_stopping_rounds=50,
        )
    return factory


def make_xgboost(params: dict, n_estimators: int | None = None):
    def factory():
        return QuietXGBClassifier(
            n_estimators=n_estimators or params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            min_child_weight=params["min_child_weight"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            reg_lambda=params["reg_lambda"],
            reg_alpha=params["reg_alpha"],
            gamma=params["gamma"],
            tree_method="hist",
            verbosity=0,
            random_state=SEED,
            n_jobs=-1,
            early_stopping_rounds=50,
        )
    return factory


def tune_catboost(trial):
    return {
        "iterations": trial.suggest_int("iterations", 300, 1200, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "depth": trial.suggest_int("depth", 3, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 50.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 300, step=10),
        "rsm": trial.suggest_float("rsm", 0.5, 1.0),
    }


def tune_lightgbm(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 7, 255),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-3, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 50.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 400, step=10),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.3, 1.0),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 1e-3, 0.5, log=True),
    }


def tune_xgboost(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 30.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.001, 5.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.001, 1.0, log=True),
    }


TUNERS = {
    "catboost": (make_catboost, tune_catboost),
    "lightgbm": (make_lightgbm, tune_lightgbm),
    "xgboost": (make_xgboost, tune_xgboost),
}


def best_iteration(model) -> int | None:
    """Число деревьев, на котором early stopping остановился.

    Имена поля разные: у CatBoost и LightGBM метод с подчёркиванием, у XGBoost —
    без. При вырожденном `eval_set` поля может не быть вовсе, тогда `None`, и
    число деревьев берётся из найденных Optuna.
    """
    for attribute in ("get_best_iteration", "best_iteration_", "best_iteration"):
        value = getattr(model, attribute, None)
        if callable(value):
            value = value()
        if value:
            return int(value)
    return None


def inner_eval_fold(outer_start: str) -> tuple[np.ndarray, np.ndarray]:
    """Последние `EARLY_STOP_DAYS` дней перед внешним фолдом — только для early stopping."""
    outer_start_ts = pd.Timestamp(outer_start)
    cut = outer_start_ts - pd.Timedelta(days=EARLY_STOP_DAYS)
    train_mask = (root < cut).values
    eval_mask = ((root >= cut) & (root < outer_start_ts)).values
    return train_mask, eval_mask


EARLY_STOP_DAYS = 4


def nested_objective(make_factory, suggest_params, X: pd.DataFrame, y: np.ndarray,
                     X_by_fold: list[pd.DataFrame] | None = None) -> float:
    """Средний P@R0.7 по трём внешним фолдам.

    `X_by_fold` обязателен для наборов с fold-local признаками: если подбор
    смотрел бы на матрицу, собранную по всему train, а оценка — на
    fold-local, то сравнение перестало бы честным, и словарь биграмм или
    target encoding UA подглядывали бы в валидационный фолд.
    """
    fold_scores = []
    for i, (start, end) in enumerate(VALID_FOLDS, 1):
        train_mask, valid_mask = fold_masks(start, end)
        fold_X = X if X_by_fold is None else X_by_fold[i - 1]
        early_stop_train, early_stop_eval = inner_eval_fold(start)
        early_stop_train = early_stop_train & train_mask

        params = suggest_params()

        # Число деревьев определяем на внутреннем фолде — единственное место,
        # где early stopping смотрит в прошлое, не заглядывая в valid фолда.
        probe = make_factory(params)()
        probe.fit(
            fold_X[early_stop_train], y[early_stop_train],
            eval_set=[(fold_X[early_stop_eval], y[early_stop_eval])],
        )
        trees = best_iteration(probe)

        model = fit_model(make_factory(params, trees), fold_X, y, train_mask, None)
        p_valid = model.predict_proba(fold_X[valid_mask])[:, 1]
        fold_scores.append(precision_at_recall(y[valid_mask], p_valid))
    return float(np.mean(fold_scores))


def run_study(name: str, suggest_params, X: pd.DataFrame, y: np.ndarray,
              X_by_fold: list[pd.DataFrame] | None = None) -> dict:
    """Полный цикл Optuna для одной модели на одной конфигурации признаков."""
    make_factory = TUNERS[name][0]

    def objective(trial):
        score = nested_objective(make_factory, lambda: suggest_params(trial), X, y, X_by_fold)
        print(f"  {name} trial {trial.number + 1}/{N_TRIALS}: {score:.4f}", flush=True)
        return score

    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"{name}: best CV mean P@R0.7 = {study.best_value:.4f}")
    print(f"{name}: best params = {study.best_params}")
    return {"params": study.best_params, "value": study.best_value}

## Сборка матриц и лестница гипотез

Признаки, зависящие от обучающей выборки — словарь биграмм и target encoding
UA — пересобираются для каждого фолда только по его train-части. Граф от
фолда не зависит: он построен по всему файлу событий, и это именно то, что
делает его переносимым на тест.

Каждая ступень лестницы — полный цикл: 30 проб Optuna для всех трёх моделей на
**одном и том же** наборе признаков, затем 3-фолдовая оценка и блендинг. Росток
обслуживания переносится в оценку, а не в подбор, иначе сравнение моделей
становится нечестным.

In [7]:
MODELS = ["catboost", "lightgbm", "xgboost"]

# Порядок гипотез фиксирован: каждая следующая проверяется на наборе, который
# накопился из ПРИНЯТЫХ ступеней, а не на полном наборе всех групп.
LADDER_STEPS = ["graph", "temporal", "bigram", "ua_semantic"]
BASE_GROUPS = {"basic": True, "temporal": False, "bigram": False,
               "graph": False, "ua_semantic": False}
LADDER = [("base", BASE_GROUPS)] + [(f"+{key}", None) for key in LADDER_STEPS]


def build_matrix(groups: dict, fit_mask: np.ndarray | None = None):
    """Признаки для всех train-кук; fold-local артефакты — по `fit_mask`."""
    engineer = FeatureEngineer(groups)
    meta = train if fit_mask is None else train[fit_mask]
    target = train.target.values if fit_mask is None else train.target.values[fit_mask]
    engineer.fit(events, meta, target)
    return engineer.transform(ev_train, train), engineer


def cv_evaluate_multi(factory, X: pd.DataFrame, y: np.ndarray,
                      name: str, X_by_fold: list[pd.DataFrame] | None = None) -> dict:
    """`cv_evaluate`, но с отдельной матрицей признаков для каждого фолда."""
    if X_by_fold is None:
        return cv_evaluate(factory, X, y, name)

    per_fold, oof_rank, oof_y = [], [], []
    for i, (start, end) in enumerate(VALID_FOLDS, 1):
        train_mask, valid_mask = fold_masks(start, end)
        fold_X = X_by_fold[i - 1]
        model = fit_model(factory, fold_X, y, train_mask, valid_mask)
        p_valid = model.predict_proba(fold_X[valid_mask])[:, 1]
        p_train = model.predict_proba(fold_X[train_mask])[:, 1]
        per_fold.append({
            "fold": i, "n_valid": int(valid_mask.sum()), "n_bots": int(y[valid_mask].sum()),
            "P@R0.7": precision_at_recall(y[valid_mask], p_valid),
            "AUC": roc_auc_score(y[valid_mask], p_valid),
            "train P@R0.7": precision_at_recall(y[train_mask], p_train),
        })
        oof_rank.append(to_rank(p_valid))
        oof_y.append(y[valid_mask])

    table = pd.DataFrame(per_fold).set_index("fold")
    table["gap"] = table["train P@R0.7"] - table["P@R0.7"]
    oof_rank = np.concatenate(oof_rank)
    oof_y = np.concatenate(oof_y)
    return {"name": name, "table": table,
            "pooled_oof": precision_at_recall(oof_y, oof_rank),
            "oof_rank": oof_rank, "oof_y": oof_y}


def run_rung(name: str, groups: dict, studies: dict | None = None) -> dict:
    """Полный цикл для одной конфигурации признаков.

    `studies` передаётся при переиспользовании, но по умолчаче каждый вызов
    подбирает параметры заново: переносить параметры между конфигурациями
    признаков нельзя, иначе сравнение моделей нечестно.
    """
    t0 = time.time()
    X_full, _ = build_matrix(groups)
    X_folds = [build_matrix(groups, fold_masks(s, e)[0])[0] for s, e in VALID_FOLDS]
    print(f"\n=== ступень {name}: {X_full.shape[1]} признаков "
          f"(сборка {time.time() - t0:.0f} с) ===")

    if studies is None:
        studies = {}
        for model_name in MODELS:
            suggest = TUNERS[model_name][1]
            studies[model_name] = run_study(model_name, suggest, X_full, y, X_folds)

    results = {}
    for model_name in MODELS:
        make_factory = TUNERS[model_name][0]
        factory = make_factory(studies[model_name]["params"])
        results[model_name] = cv_evaluate_multi(factory, X_full, y, model_name, X_folds)
        low, _, high = bootstrap_pooled(results[model_name])
        print(f"  {model_name:9s} pooled={results[model_name]['pooled_oof']:.4f} "
              f"min_fold={results[model_name]['table']['P@R0.7'].min():.4f} "
              f"gap={results[model_name]['table']['gap'].iloc[2]:.4f} "
              f"[90% {low:.4f}..{high:.4f}]")

    blend = np.column_stack([results[m]["oof_rank"] for m in MODELS]).mean(axis=1)
    oof_y = results[MODELS[0]]["oof_y"]
    best_single = max(MODELS, key=lambda m: results[m]["pooled_oof"])
    return {
        "name": name, "groups": groups, "studies": studies, "results": results,
        "X": X_full, "oof_y": oof_y,
        "blend_oof": precision_at_recall(oof_y, blend),
        "blend_rank": blend,
        "best_single": best_single,
        "single_oof": results[best_single]["pooled_oof"],
        "min_fold": min(results[m]["table"]["P@R0.7"].min() for m in MODELS),
        "seconds": time.time() - t0,
    }

## Stage 1 — контрольный прогон на базовых признаках

Проверяем, что рефакторинг ничего не сломал. Эталон: прежний CatBoost давал
pooled OOF 0.6942 на 28 признаках при **узком** поисковом пространстве
(`depth` 3–6, `learning_rate` до 0.15, `min_data_in_leaf` от 50). Здесь
пространство расширено, поэтому 0.6942 — не точное число, а нижняя граница:
корректный контроль — ниже не откатиться.

In [8]:
stage1 = run_rung("base", LADDER[0][1])

print("\nper-fold:")
print(pd.DataFrame({
    m: stage1["results"][m]["table"]["P@R0.7"] for m in MODELS
}).round(4).to_string())

BASELINE_OOF = 0.6942
catboost_oof = stage1["results"]["catboost"]["pooled_oof"]
print(f"\nэталон (узкое пространство): {BASELINE_OOF:.4f}")
print(f"catboost (расширенное пространство): {catboost_oof:.4f} "
      f"({catboost_oof - BASELINE_OOF:+.4f})")
assert catboost_oof >= BASELINE_OOF - 0.02, (
    f"рефакторинг ухудшил базу: {catboost_oof:.4f} против {BASELINE_OOF:.4f}"
)
print("контроль пройден: база не деградировала")

[I 2026-10-03 22:16:15,334] A new study created in memory with name: no-name-d2e84dae-49b2-4adc-bb6f-e1a8babfc783



=== ступень base: 28 признаков (сборка 53 с) ===


  catboost trial 1/30: 0.6479


[I 2026-10-03 22:16:19,785] Trial 0 finished with value: 0.6479161659857406 and parameters: {'iterations': 800, 'learning_rate': 0.0852084800119005, 'depth': 9, 'l2_leaf_reg': 8.428303845098709, 'min_data_in_leaf': 130, 'rsm': 0.8229470565333281}. Best is trial 0 with value: 0.6479161659857406.


  catboost trial 2/30: 0.5847


...

[I 2026-10-03 22:22:56,799] Trial 29 finished with value: 0.6578730854739346 and parameters: {'n_estimators': 400, 'learning_rate': 0.014259969289215357, 'max_depth': 8, 'min_child_weight': 0.83064773084749, 'subsample': 0.6433933002474419, 'colsample_bytree': 0.5128729016420502, 'reg_lambda': 0.15813163174643585, 'reg_alpha': 0.006087160393551525, 'gamma': 0.17010614720525835}. Best is trial 14 with value: 0.6643307961101931.


xgboost: best CV mean P@R0.7 = 0.6643
xgboost: best params = {'n_estimators': 600, 'learning_rate': 0.017991523996077896, 'max_depth': 7, 'min_child_weight': 0.526095209569724, 'subsample': 0.6512400974600892, 'colsample_bytree': 0.5259994315790945, 'reg_lambda': 0.5303638119497459, 'reg_alpha': 0.003518950580485202, 'gamma': 0.21795633816792293}


  catboost  pooled=0.6871 min_fold=0.6522 gap=0.2320 [90% 0.6170..0.7480]


  lightgbm  pooled=0.6515 min_fold=0.5882 gap=0.2503 [90% 0.5585..0.7371]


  xgboost   pooled=0.6546 min_fold=0.6027 gap=0.2539 [90% 0.5861..0.7353]

per-fold:
      catboost  lightgbm  xgboost
fold                             
1       0.6692    0.6397   0.6350
2       0.6522    0.5882   0.6027
3       0.7205    0.7197   0.7386

эталон (узкое пространство): 0.6942
catboost (расширенное пространство): 0.6871 (-0.0071)
контроль пройден: база не деградировала


## Вложенный стэкинг и финальные предсказания

Сравниваются два способа объединения, оба — честные, без утечки:
rank-average блендинг и стэкинг с логистической регрессией в качестве
мета-learner'а. Мета-learner обучается на внутренних временных фолдах
train-части внешнего окна, поэтому метки внешнего валидационного фолда он не
видел никогда.

In [9]:
def inner_folds_for(outer_start: str, n_chunks: int = 3):
    """Временные фолды внутри train-части внешнего окна, строго в прошлом."""
    days = sorted(root.dt.normalize().unique())
    past_days = [d for d in days if d < pd.Timestamp(outer_start)]
    if len(past_days) < n_chunks + 1:
        return []
    blocks = np.array_split(np.array(past_days), n_chunks + 1)
    folds = []
    for block in blocks[1:]:
        chunk_start = block[0]
        chunk_end = block[-1] + pd.Timedelta(days=1)
        folds.append(((root < chunk_start).values,
                      ((root >= chunk_start) & (root < chunk_end)).values))
    return folds


def ranked_oof(factory, fold_X: pd.DataFrame, y: np.ndarray, folds) -> np.ndarray:
    """OOF-столбец в рангах; `to_rank` применяется внутри каждого фолда.

    Ранги, а не сырые вероятности: у моделей с разной регуляризацией шкалы
    несопоставимы, и линейная комбинация таких колонок отражала бы различие
    калибровки, а не качество.
    """
    scores = np.full(len(y), np.nan)
    for train_mask, eval_mask in folds:
        model = fit_model(factory, fold_X, y, train_mask, eval_mask)
        scores[eval_mask] = to_rank(model.predict_proba(fold_X[eval_mask])[:, 1])
    return scores


def nested_stack(factories: dict, X_folds: list[pd.DataFrame],
                 y: np.ndarray) -> tuple[np.ndarray, dict]:
    """Честные OOF-предсказания вложенного стэкинга с мета-learner'ом."""
    names = list(factories)
    meta_oof, weights = [], None

    for i, (start, end) in enumerate(VALID_FOLDS, 1):
        train_mask, valid_mask = fold_masks(start, end)
        inner_folds = inner_folds_for(start)
        assert inner_folds, f"не хватило истории для внутренних фолдов: {start}"
        fold_X = X_folds[i - 1]

        inner_matrix = np.column_stack([
            np.nan_to_num(ranked_oof(factories[name], fold_X, y, inner_folds))
            for name in names
        ])
        meta = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
        meta.fit(inner_matrix[train_mask], y[train_mask])

        outer_matrix = np.column_stack([
            to_rank(fit_model(factories[name], fold_X, y, train_mask, None)
                    .predict_proba(fold_X[valid_mask])[:, 1])
            for name in names
        ])
        meta_oof.append(meta.predict_proba(outer_matrix)[:, 1])
        weights = dict(zip(names, meta.coef_[0].round(3)))

    return np.concatenate(meta_oof), weights


def final_test_scores(rung: dict, X_test: pd.DataFrame) -> dict[str, np.ndarray]:
    """Предсказания на test: блендинг и стэкинг, обученные на всех train-окнах."""
    factories = {m: TUNERS[m][0](rung["studies"][m]["params"]) for m in MODELS}
    all_train = np.ones(len(y), dtype=bool)
    test_rank = {}

    for model_name, factory in factories.items():
        model = fit_model(factory, rung["X"], y, all_train, None)
        test_rank[model_name] = to_rank(model.predict_proba(X_test)[:, 1])
        print(f"  обучена финальная модель на всех train-окнах: {model_name}")

    blend = np.column_stack(list(test_rank.values())).mean(axis=1)

    inner_folds = inner_folds_for(VALID_FOLDS[-1][0])
    meta_matrix = np.column_stack([
        np.nan_to_num(ranked_oof(factories[name], rung["X"], y, inner_folds))
        for name in MODELS
    ])
    meta = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED).fit(meta_matrix, y)
    stack = meta.predict_proba(np.column_stack(list(test_rank.values())))[:, 1]
    print(f"  веса финального мета-learner'а: {dict(zip(MODELS, meta.coef_[0].round(3)))}")
    return {"blend": blend, "stack": stack}

## Проход по лестнице гипотез

Ступень принимается, если её оценка (лучшее из блендинга и лучшей одиночной
модели) превзошла текущую более чем на `MIN_GAIN`. Иначе группа откатывается и
следующая ступень проверяется на прежнем наборе.

In [10]:
def rung_row(rung: dict, previous: float | None) -> dict:
    best = max(rung["blend_oof"], rung["single_oof"])
    return {
        "Этап": rung["name"],
        "признаков": rung["X"].shape[1],
        "лучшая модель": (f'блендинг ({", ".join(MODELS)})'
                          if rung["blend_oof"] >= rung["single_oof"]
                          else rung["best_single"]),
        "Pooled OOF": round(best, 4),
        "Blend": round(rung["blend_oof"], 4),
        "Single": round(rung["single_oof"], 4),
        "Min Fold (все)": round(rung["min_fold"], 4),
        "прирост": np.nan if previous is None else round(best - previous, 4),
        "секунд": round(rung["seconds"]),
    }


LADDER_RESULTS = [stage1]
accepted = [stage1]
current = stage1
current_groups = dict(LADDER[0][1])
rows = [rung_row(stage1, None)]

for name, _ in LADDER[1:]:
    # Каждая гипотеза проверяется на ТЕКУЩЕМ принятом наборе, а не на полном:
    # иначе откат был бы фиктивным и отклонённые группы всё равно доезжали бы
    # в следующую ступень.
    key = name[1:]
    groups = {**current_groups, key: True}
    previous = max(current["blend_oof"], current["single_oof"])
    rung = run_rung(name, groups)
    LADDER_RESULTS.append(rung)
    gain = max(rung["blend_oof"], rung["single_oof"]) - previous
    row = rung_row(rung, previous)
    row["принята"] = gain > MIN_GAIN
    rows.append(row)
    print(f"   ступень {name} проверялась на {rung['X'].shape[1]} признаках: "
          f"{sorted(k for k, v in groups.items() if v)}")
    if row["принята"]:
        current_groups = groups
        accepted.append(rung)
        current = rung
    else:
        print(f"\nступень {name} откатена: прирост {gain:+.4f} <= {MIN_GAIN}")

LADDER_TABLE = pd.DataFrame(rows)
print("\n=== лестница гипотез ===")
print(LADDER_TABLE.to_string(index=False))

best_rung = accepted[-1]
print(f"\nпринятые ступени: {[r['name'] for r in accepted]}")
print(f"итоговый набор: {best_rung['name']}, признаков: {best_rung['X'].shape[1]}")

[I 2026-10-03 22:23:55,136] A new study created in memory with name: no-name-08223166-2fd5-4a64-a043-75696dbf4738



=== ступень +graph: 32 признаков (сборка 54 с) ===


  catboost trial 1/30: 0.8638


[I 2026-10-03 22:23:59,176] Trial 0 finished with value: 0.8637615046018409 and parameters: {'iterations': 800, 'learning_rate': 0.0852084800119005, 'depth': 9, 'l2_leaf_reg': 8.428303845098709, 'min_data_in_leaf': 130, 'rsm': 0.8229470565333281}. Best is trial 0 with value: 0.8637615046018409.


  catboost trial 2/30: 0.8315


...

[I 2026-10-03 23:17:21,718] Trial 29 finished with value: 0.8606406242671545 and parameters: {'n_estimators': 300, 'learning_rate': 0.016510778556230957, 'max_depth': 10, 'min_child_weight': 1.5137571004484727, 'subsample': 0.5623461147133727, 'colsample_bytree': 0.5131696922505121, 'reg_lambda': 0.07573470215273058, 'reg_alpha': 0.054912459610296435, 'gamma': 0.5645440963787056}. Best is trial 21 with value: 0.8837606837606837.


xgboost: best CV mean P@R0.7 = 0.8838
xgboost: best params = {'n_estimators': 400, 'learning_rate': 0.014542109246668088, 'max_depth': 10, 'min_child_weight': 4.7926481592371255, 'subsample': 0.5799904475824702, 'colsample_bytree': 0.6508318761435833, 'reg_lambda': 0.05469073456131362, 'reg_alpha': 0.16182337661257476, 'gamma': 0.9681861751240437}


  catboost  pooled=0.8720 min_fold=0.8444 gap=0.1537 [90% 0.8109..0.9097]


  lightgbm  pooled=0.8614 min_fold=0.8550 gap=0.0960 [90% 0.8076..0.8997]


  xgboost   pooled=0.8627 min_fold=0.8667 gap=0.1202 [90% 0.8242..0.9003]
   ступень +ua_semantic проверялась на 50 признаках: ['basic', 'bigram', 'graph', 'ua_semantic']

ступень +ua_semantic откатена: прирост -0.0027 <= 0.005

=== лестница гипотез ===
        Этап  признаков                          лучшая модель  Pooled OOF  Blend  Single  Min Fold (все)  прирост  секунд принята
        base         28                               catboost      0.6871 0.6620  0.6871          0.5882      NaN     459     NaN
      +graph         32                                xgboost      0.8614 0.8542  0.8614          0.8370   0.1744     386    True
   +temporal         42                               lightgbm      0.8614 0.8563  0.8614          0.8248   0.0000     710   False
     +bigram         47 блендинг (catboost, lightgbm, xgboost)      0.8746 0.8746  0.8720          0.8296   0.0132    1132    True
+ua_semantic         50 блендинг (catboost, lightgbm, xgboost)      0.8720 0.8720  0.8720  

## Финальный выбор и сабмит

На принятом наборе признаков сравниваются блендинг и вложенный стэкинг, оба
обученные на всех train-окнах. Композитный скор реранжится порядковыми
рангами: метрика не умеет делить равные значения, поэтому дискретизация
реально меняет оценку, а `ordinal` не вносит ничьих.

Сабмит записывается только если локальная оценка выше `PROMOTE_THRESHOLD`.
Текущий защищаемый результат — публичный LB 0.734.

In [11]:
X_best_train = best_rung["X"]
_, best_engineer = build_matrix(best_rung["groups"])
X_best_test = best_engineer.transform(ev_test, test)

assert list(X_best_test.columns) == list(X_best_train.columns), "схемы train и test различаются"
assert X_best_train.notna().all().all() and X_best_test.notna().all().all(), "в признаках пропуски"

factories_best = {m: TUNERS[m][0](best_rung["studies"][m]["params"]) for m in MODELS}
X_folds_best = [build_matrix(best_rung["groups"], fold_masks(s, e)[0])[0]
                for s, e in VALID_FOLDS]

stack_oof, stack_weights = nested_stack(factories_best, X_folds_best, y)
blend_oof_best = precision_at_recall(best_rung["oof_y"], best_rung["blend_rank"])

ENSEMBLES = {
    "blend": {"oof": best_rung["blend_rank"], "score": blend_oof_best},
    "stack": {"oof": stack_oof, "score": precision_at_recall(best_rung["oof_y"], stack_oof)},
}
print("=== схемы объединения на принятом наборе ===")
for label, values in ENSEMBLES.items():
    print(f"  {label:6s} pooled OOF = {values['score']:.4f}")
print(f"  веса мета-learner'а: {stack_weights}")
print(f"  потолок стэкинга с утечкой (диагностика) = "
      f"{precision_at_recall(best_rung['oof_y'], np.maximum(best_rung['blend_rank'], stack_oof)):.4f}")

ensemble_name = max(ENSEMBLES, key=lambda k: ENSEMBLES[k]["score"])
ensemble = ENSEMBLES[ensemble_name]
print(f"\nпобедитель: {ensemble_name}, pooled OOF = {ensemble['score']:.4f}")

test_scores = final_test_scores(best_rung, X_best_test)[ensemble_name]
# Порядковые ранги: равных значений метрика не разделяет, а jitter вносил бы шум.
test_scores = to_rank(test_scores)

def write_submission(scores: np.ndarray, path: str) -> pd.DataFrame:
    """Оформление сабмита со всеми ассертами формата."""
    submission = test[["cookie_id"]].copy()
    submission["score"] = np.clip(scores, 0.0, 1.0)

    template = pd.read_csv("data/sample_submission.csv")
    assert list(submission.columns) == ["cookie_id", "score"], "неверные колонки"
    assert set(submission.cookie_id) == set(template.cookie_id), "не совпадают cookie_id"
    assert set(submission.cookie_id) == set(test.cookie_id), "не совпадают cookie_id с test"
    assert len(submission) == len(test) == len(template), "не совпадает число строк"
    assert submission.cookie_id.is_unique, "cookie_id не уникальны"
    assert submission.score.between(0.0, 1.0).all(), "score вне [0, 1]"
    assert submission.score.nunique() > 0.99 * len(submission), (
        f"дискретизация score: {submission.score.nunique()} уникальных из {len(submission)}"
    )
    submission.to_csv(path, index=False)
    return submission


print(f"\nлокальный OOF победителя: {ensemble['score']:.4f} "
      f"(порог сабмита {PROMOTE_THRESHOLD})")
if ensemble["score"] > PROMOTE_THRESHOLD:
    submission = write_submission(test_scores, "submission.csv")
    print("submission.csv ЗАМЕНЁН")
else:
    path = f"submission_v3_{best_rung['name']}_{ensemble_name}.csv"
    submission = write_submission(test_scores, path)
    print(f"порог не достигнут — submission.csv НЕ ТРОНУТ, результат в {path}")

print(f"saved: {submission.shape}, уникальных score: {submission.score.nunique()}")
print(submission.head())

=== схемы объединения на принятом наборе ===
  blend  pooled OOF = 0.8746
  stack  pooled OOF = 0.7836
  веса мета-learner'а: {'catboost': np.float64(1.047), 'lightgbm': np.float64(1.041), 'xgboost': np.float64(1.065)}
  потолок стэкинга с утечкой (диагностика) = 0.8746

победитель: blend, pooled OOF = 0.8746


  обучена финальная модель на всех train-окнах: catboost


  обучена финальная модель на всех train-окнах: lightgbm


  обучена финальная модель на всех train-окнах: xgboost


  веса финального мета-learner'а: {'catboost': np.float64(0.745), 'lightgbm': np.float64(0.742), 'xgboost': np.float64(0.759)}

локальный OOF победителя: 0.8746 (порог сабмита 0.75)
submission.csv ЗАМЕНЁН
saved: (4909, 2), уникальных score: 4909
             cookie_id     score
0  ck_315fb710a0e371e7  0.061112
1  ck_a76ee3b3e3e522fd  0.852108
2  ck_94c9a4d382689e82  0.559992
3  ck_8eaf9509ad9462a0  0.036667
4  ck_9a88a5a989cb5bc6  0.282746


## Диагностика переноса на test

Pooled OOF ничего не говорит о том, что будет на тестовых окнах: там нет
меток. Поэтому проверяем три вещи, которые ловят сдвиг распределения.

1. **Важность.** Если бы прыжок давал один-единственный признак, это был бы
   кандидат на утечку. Распределение важности проверяем по группам.
2. **Операционная точка.** Доля test-кук выше OOF-порога при recall 0.70
   должна совпадать с долей на OOF. Ранжирование делает score равномерным,
   поэтому сравнивать надо именно долю выше порога, а не среднее значение.
3. **Стабильность по окнам.** Разброс доли по семи тестовым окнам покажет,
   не ломается ли модель на отдельных днях.

In [12]:
importance = {}
all_train_mask = np.ones(len(y), dtype=bool)
for model_name in MODELS:
    model = fit_model(factories_best[model_name], X_best_train, y, all_train_mask, None)
    importance[model_name] = pd.Series(model.feature_importances_,
                                       index=X_best_train.columns)


def feature_group(column: str) -> str:
    if column.startswith(("g_item", "g_catloc")):
        return "graph"
    if column.startswith("bg_"):
        return "bigram"
    return "basic"


importance_share = pd.DataFrame(
    {name: values / values.sum() for name, values in importance.items()})
group_labels = pd.Series([feature_group(c) for c in importance_share.columns],
                         index=importance_share.columns)
print("=== доля суммарной важности по группам, % ===")
print((100 * importance_share.groupby(group_labels).sum().mean()).round(1).to_string())

print("\n=== топ-8 признаков каждой модели ===")
for name, values in importance.items():
    top = values.sort_values(ascending=False).head(8)
    print(f"{name:9s} " + ", ".join(f"{c}={values[c] / values.sum():.3f}" for c in top.index))

# Операционная точка OOF: минимальный порог, при котором recall ещё >= 0.70.
oof_scores, oof_labels = best_rung["blend_rank"], best_rung["oof_y"]
target_positives = int(np.ceil(0.70 * oof_labels.sum()))
operating_threshold = float(np.sort(oof_scores)[::-1][target_positives - 1])
oof_flag_rate = float((oof_scores > operating_threshold).mean())

test_flag_rate = float((submission.score > operating_threshold).mean())
per_window = submission.merge(test[["cookie_id", "window_start_ts"]], on="cookie_id") \
    .groupby("window_start_ts").score.apply(lambda s: (s > operating_threshold).mean())

print(f"\n=== операционная точка ===")
print(f"порог OOF при recall 0.70 = {operating_threshold:.4f}")
print(f"доля выше порога: OOF {oof_flag_rate:.4f} | test {test_flag_rate:.4f} "
      f"(разница {test_flag_rate - oof_flag_rate:+.4f})")
print(f"ожидаемая доля на test при сохранении точности: "
      f"{target_positives / ensemble['score'] / len(test):.4f}")
print("доля выше порога по тестовым окнам:")
print("  " + str(per_window.round(4).to_dict()))
assert abs(test_flag_rate - oof_flag_rate) < 0.02, (
    f"распределение тестовых скоров уело от OOF: {test_flag_rate:.4f} против {oof_flag_rate:.4f}"
)
print("перенос в пределах нормы")

=== доля суммарной важности по группам, % ===
catboost   NaN
lightgbm   NaN
xgboost    NaN

=== топ-8 признаков каждой модели ===
catboost  g_item_mean_rel=0.100, pointer_std=0.095, nuniq_item_location=0.056, frac_gap_gt60s=0.049, gap_entropy=0.045, nuniq_item_category=0.044, gap_median=0.035, gap_cv=0.029
lightgbm  g_item_mean_rel=0.076, gap_median=0.075, mean_hod=0.071, gap_p90=0.056, frac_gap_gt60s=0.044, gap_cv=0.041, g_catloc_mean_rel=0.037, nuniq_item_location=0.037
xgboost   g_item_mean_rel=0.125, pointer_nuniq=0.071, pointer_std=0.062, frac_gap_gt60s=0.045, nuniq_item_location=0.044, nuniq_item_category=0.038, g_item_max_rel=0.038, nuniq_item_id=0.034

=== операционная точка ===
порог OOF при recall 0.70 = 0.9451
доля выше порога: OOF 0.0552 | test 0.0550 (разница -0.0002)
ожидаемая доля на test при сохранении точности: 0.0666
доля выше порога по тестовым окнам:
  {Timestamp('2026-04-20 00:00:00'): 0.0552, Timestamp('2026-04-21 00:00:00'): 0.051, Timestamp('2026-04-22 00:00:00'